#### RAG Data Ingestion to vectorDb Pipeline


In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

d:\RAG Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages ")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
    print(f"\nTotal documents loaded so far: {len(all_documents)}")

    return all_documents
    # Process all PDFs in data directory
all_docs = process_all_pdfs("../data")

Found 5 PDF files to process

Processing: 613  (1).pdf
Loaded 10 pages 

Processing: Swaraj_Waykar .pdf
Loaded 1 pages 

Processing: AdmitCard (1).pdf
Loaded 1 pages 

Processing: Swaraj_Shahaji_Waykar_CV.pdf
Loaded 1 pages 

Processing: Swaraj_Waykar_Resume.pdf
Loaded 1 pages 

Total documents loaded so far: 14


In [6]:
all_docs

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-05-31T16:31:13+05:30', 'author': 'Swaraj Waykar', 'moddate': '2025-05-31T16:31:13+05:30', 'title': 'Microsoft Word - 613.docx', 'source': '..\\data\\613  (1).pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': '613  (1).pdf', 'file_type': 'pdf'}, page_content="AI TRAVEL BUDDY: EMOTION DETECTION AND \nCONVERSATIONAL AI FOR ENHANCING SOLO \nTRAVEL EXPERIENCES. \n \nNilesh Korade1 , Swaraj Waykar2 , Archit Waghmode3 ,Shweta Tate4 \n \nDept. of Computer Engineering JSPM’s Rajarshi Shahu College of Engineering Pune, India \nnilesh.korade.ml@gmail.com, swarajwaykar8@gmail.com, arc.hit.mode07@gmail.com, \nshwetatate30@jspmrscoe.edu.in \n \n \nAbstract. Though solo travel can be an enriching and adventurous experience, \nstaying alone (without social interactions) can sometimes enhance feelings of \nloneliness and this can be detrimental to a traveler's mental health. In this 

In [7]:
### Text splitting into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:500]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        return split_docs

    return split_docs

In [8]:
chunks = split_documents(all_docs)
chunks

Split 14 documents into 57 chunks

Example chunk:
Content: AI TRAVEL BUDDY: EMOTION DETECTION AND 
CONVERSATIONAL AI FOR ENHANCING SOLO 
TRAVEL EXPERIENCES. 
 
Nilesh Korade1 , Swaraj Waykar2 , Archit Waghmode3 ,Shweta Tate4 
 
Dept. of Computer Engineering JSPM’s Rajarshi Shahu College of Engineering Pune, India 
nilesh.korade.ml@gmail.com, swarajwaykar8@gmail.com, arc.hit.mode07@gmail.com, 
shwetatate30@jspmrscoe.edu.in 
 
 
Abstract. Though solo travel can be an enriching and adventurous experience, 
staying alone (without social interactions) can so...
Metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-05-31T16:31:13+05:30', 'author': 'Swaraj Waykar', 'moddate': '2025-05-31T16:31:13+05:30', 'title': 'Microsoft Word - 613.docx', 'source': '..\\data\\613  (1).pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': '613  (1).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-05-31T16:31:13+05:30', 'author': 'Swaraj Waykar', 'moddate': '2025-05-31T16:31:13+05:30', 'title': 'Microsoft Word - 613.docx', 'source': '..\\data\\613  (1).pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': '613  (1).pdf', 'file_type': 'pdf'}, page_content="AI TRAVEL BUDDY: EMOTION DETECTION AND \nCONVERSATIONAL AI FOR ENHANCING SOLO \nTRAVEL EXPERIENCES. \n \nNilesh Korade1 , Swaraj Waykar2 , Archit Waghmode3 ,Shweta Tate4 \n \nDept. of Computer Engineering JSPM’s Rajarshi Shahu College of Engineering Pune, India \nnilesh.korade.ml@gmail.com, swarajwaykar8@gmail.com, arc.hit.mode07@gmail.com, \nshwetatate30@jspmrscoe.edu.in \n \n \nAbstract. Though solo travel can be an enriching and adventurous experience, \nstaying alone (without social interactions) can sometimes enhance feelings of \nloneliness and this can be detrimental to a traveler's mental health. In this 

### Embeddings and VectorStoreDB



In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [10]:
class EmbeddingManager:
    """Handles embedding generation using Sentence Transformers"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        Args:
            model_name (str): huggingface model name for sentence Embeddings
            """
        self.model_name = model_name
        self.model=None
        self.load_model()

    def load_model(self):  
        """Load sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Successfully loaded embedding model. Embedding dimensionality: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")  
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
            texts (List[str]): List of text strings to embed    
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    ### Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager
           
      


Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2695.13it/s]


Successfully loaded embedding model. Embedding dimensionality: 384


### VectorStore

In [11]:
import os
from pathlib import Path
import chromadb
import uuid
import numpy as np
from typing import List, Any
class VectorStore:
    """Manages vector storage and retrieval using ChromaDB"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store
        Args:
            collection_name (str): Name of the ChromaDB collection to use
            persist_directory (str): Directory where ChromaDB will persist data
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialize_store()
    def initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #create persist chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            #get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF embeddings for RAG project"})
            print(f"Initialized VectorStore collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing VectorStore ChromaDB: {e}")
            raise    

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store    

Initialized VectorStore collection: pdf_documents
Existing documents in collection: 0


In [12]:
chunks

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-05-31T16:31:13+05:30', 'author': 'Swaraj Waykar', 'moddate': '2025-05-31T16:31:13+05:30', 'title': 'Microsoft Word - 613.docx', 'source': '..\\data\\613  (1).pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': '613  (1).pdf', 'file_type': 'pdf'}, page_content="AI TRAVEL BUDDY: EMOTION DETECTION AND \nCONVERSATIONAL AI FOR ENHANCING SOLO \nTRAVEL EXPERIENCES. \n \nNilesh Korade1 , Swaraj Waykar2 , Archit Waghmode3 ,Shweta Tate4 \n \nDept. of Computer Engineering JSPM’s Rajarshi Shahu College of Engineering Pune, India \nnilesh.korade.ml@gmail.com, swarajwaykar8@gmail.com, arc.hit.mode07@gmail.com, \nshwetatate30@jspmrscoe.edu.in \n \n \nAbstract. Though solo travel can be an enriching and adventurous experience, \nstaying alone (without social interactions) can sometimes enhance feelings of \nloneliness and this can be detrimental to a traveler's mental health. In this 

In [15]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vector_store.add_documents(chunks,embeddings)

Generating embeddings for 57 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.09it/s]

Generated embeddings with shape: (57, 384)
Adding 57 documents to vector store...
Successfully added 57 documents to vector store
Total documents in collection: 57


### Retriever Pipeline From VectorStore

In [16]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vector_store,embedding_manager)

In [17]:
rag_retriever

In [24]:

rag_retriever.retrieve("What is Emotion Detection?")

Retrieving documents for query: 'What is Emotion Detection?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.33it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_3de3d124_15',
  'content': 'architecture for detecting face features corresponding to emotions. The ResNet50 first \ndetects micro-expressions from all frames, i.e., features such as eye openness, \neyebrows position, mouth shape etc. to obtain their emotional state [4]. We then use \nthe model to classify these features into one of seven pre-defined emotions \n(happiness, sadness, anger, surprise, disgust, fear or neutral). After extracting the \nfeatures, it maps a probability score for all the emotions where emotion with max \nvalue is considered as detected emotion in a particular frame. Emotion Confirmation \n3.4 Emotion Confirmation: To improve accuracy and ensure that the user trusts \nthe system, you can ask if the detected emotion is correct. This is a resize step, which \nis optional but recommended because it allows for corrections by the user in case of \nmisclassifications. By confirming the emotions sensed, emotion confirmation',
  'metadata': {'file_type': '

In [25]:
rag_retriever.retrieve("Conversational AI with Natural Language Processing")

Retrieving documents for query: 'Conversational AI with Natural Language Processing'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 67.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_7195b751_10',
  'content': 'processing into voice assistant chatbots. Their study emphasized the importance of \ncontext-aware dialogue and emotional nuance in creating more meaningful \ninteractions, especially for applications in travel assistance.In paper [8], S. Anand and \nA.M.A. Sai proposed a smart tourism system using AI-based chatbots for Indian \ncities. Their system simplified travel planning by offering localized support. In paper \n[9], Ayo et al. developed LUNTIAN, a chatbot companion designed to provide \nemotionally aware interactions. In paper [10], N.B. Korade and colleagues explored \nthe use of OpenAI embeddings for sentence similarity detection, enhancing \nconversational relevance in AI systems. In paper [11], further work by Korade N.B. \nintroduced NLP techniques for duplicate question detection to optimize \nresponsiveness in conversational agents. In paper [12], convolutional neural networks \n(CNNs) originally designed for stock price forecasting

### RAG Pipeline- VectorDB To LLM Output Generation

In [32]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [33]:
answer=rag_simple("What is Emotion Detection?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Emotion Detection?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.37it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Emotion detection is the process of identifying and classifying emotions from facial expressions using a computer vision model, such as a CNN-based model, to analyze features like eye openness, eyebrows position, mouth shape, etc. and map a probability score for pre-defined emotions (happiness, sadness, anger, surprise, disgust, fear, or neutral).


In [34]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced(" Voice Assistance and Emotion-Based Chat ", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: ' Voice Assistance and Emotion-Based Chat '
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.27it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: The AI Travel Buddy features a Voice Assistance and Emotion-Based Chat, utilizing Natural Language Processing (NLP) and Text-to-Speech (TTS) modules to detect emotions and provide tailored, emotive responses in real-time, enhancing user engagement and travel satisfaction.
Sources: [{'source': '613  (1).pdf', 'page': 7, 'score': 0.2243586778640747, 'preview': 'mood detected but engaging and organic sounding. Emotion-Aware Response A core \npart of this functionality is Natural Language Processing (NLP), which allows the AI \nTravel Buddy to craft tailored, emotive responses in-the-moment based on what it \nimmediately gleans from traveler input. The AI simpl...'}, {'source': '613  (1).pdf', 'page': 0, 'score': 0.17296528816223145, 'preview': 'using the emotion-aware voice assistant. Using the latest CNN architecture for \nemotion detection, and Conversational model for dynamic answers, the AI \nTravel Buddy is a companion to travelers, delivering emotional support to users \non 

In [38]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is Emotion detection?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What is Emotion detection?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.27it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
architecture for detecting face features corresponding to emotions. The ResNet50 first 
detects micro-expressions from all frames, i.e., features such as eye openness, 
eyebrows position, mouth shape etc. to obtain their emotional state [4]. We then u

se 
the model to classify these features into one of seven pre-defined emotions 
(happiness, sadness, anger, surprise, disgust, fear or neutral). After extracting the 
features, it maps a probability score for all the emotions where emotion with max 
value is considered as detected emotion in a particular frame. Emotion Confirmation 
3.4 Emotion Confirmation: To improve accuracy and ensure that the user trusts 
the system, you can ask if the detected emotion is correct. This is a resize step, which 
is optional but recommended because it allows for corrections by the user in case of 
misclassifications. By confirming the emotions sensed, emotion confirmation

Fig.1 Methodology 
3.1 Video Input and Frame Extraction: The AI Travel Buddy system first uses a 
camera either integrated into the car or linked through the mobile device to capture 
real-time video of the face of the user. This video feed is the main input for emotion 
detection. In order to analyze this video, it is divided int